# R check

Run `00_start_here.ipynb` first — this reads the Parquet file it writes. If VS Code asks for a kernel, pick **R**.

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(arrow)
  library(duckdb)
})
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| dplyr", format(packageVersion("dplyr")),
    "| duckdb", format(packageVersion("duckdb")), "\n")

In [ ]:
root <- if (dir.exists("data")) "." else ".."
fc <- read_parquet(file.path(root, "data/raw/forecast_sample.parquet"))

fc |>
  group_by(park) |>
  summarise(
    avg_high_f = round(mean(temperature_2m_max), 1),
    rainy_days = sum(precipitation_probability_max >= 50, na.rm = TRUE)
  )

DuckDB works from R as well — here it queries the Parquet file directly:

In [ ]:
con <- dbConnect(duckdb())
dbGetQuery(con, sprintf(
  "SELECT park, max(wind_speed_10m_max) AS windiest_mph FROM read_parquet('%s') GROUP BY park",
  file.path(root, "data/raw/forecast_sample.parquet")
))
dbDisconnect(con, shutdown = TRUE)